# Importat librerías y leer datos

In [ ]:
import pandas as pd
import json
import folium
import geopandas as gpd
from shapely.geometry import LineString, Point
from shapely import wkt
import math
import time
from tqdm import tqdm  

# Cargar datos de espiras electromagnéticas (el JSON que subiste)
with open("../data/punts-mesura-trafic-espires-electromagnetiques-puntos-medida-trafico-espiras-ele.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Inspeccionar estructura
print(data[0].keys())


Los datos provienen de mediciones de espiras en la ciudad de Valencia, del día 22/10/2025 entre las 12:00 y las 13:00 horas.

# Extraer datos relevantes

In [ ]:
df_espiras = pd.json_normalize(data)

print("Filas:", len(df_espiras))


df_espiras.head()

Empecemos eliminando las variables que no son de nuestro interés

In [ ]:
cols_to_drop = ["fecha_actualizacion", "last_edited_date", "created_user", "created_date", "last_edited_user", "globalid", "fiwareid", "geo_shape.type", "geo_shape.geometry.type"]
df = df_espiras.drop(columns=cols_to_drop, errors="ignore")

Por simplicidad vamos a renombrar el nombre de nuestras variables

In [ ]:
# --- Renombrar columnas clave ---
df = df.rename(columns={
    'gid' : "identificador", 
    'angulo' :  "angulo_sentido_circulacion", 
    'idpm': "id_punto_medida",
    'ih': "vehiculos_por_hora", 
    'geo_shape.geometry.coordinates': "coordenadas", 
    'geo_point_2d.lon': "longitud",
    'geo_point_2d.lat': "latitud"
})

In [ ]:
df.head()

Eliminamos los nulos en vehiculos por hora, ya que es la variable target del estudio

In [ ]:
df = df.dropna(subset=["vehiculos_por_hora"])

# Combinar datos OSM con Espiras Valencia

In [ ]:
df_osm = pd.read_csv('../data/datos_valencia_limpios.csv')

In [ ]:
df_osm["geometry"].head()

In [ ]:
# Convierte de texto (WKT) a geometría real
df_osm["geometry"] = df_osm["geometry"].apply(wkt.loads)

Dibujamos sobre el mapa las calles recopiladas de Valencia y las espiras

In [ ]:
# Crear GeoDataFrame de espiras
gdf_espiras = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitud, df.latitud),
    crs="EPSG:4326"
)


m = folium.Map(location=[39.4699, -0.3763], zoom_start=13, tiles="cartodb positron")



# --- Tramos OSM ---
for _, row in df_osm.iterrows():
    if isinstance(row.geometry, LineString):
        coords = [(lat, lon) for lon, lat in row.geometry.coords]
        folium.PolyLine(
            coords,
            color="blue",
            weight=2,
            opacity=0.6,
            popup=row.get("name", "Tramo OSM")
        ).add_to(m)

# --- Espiras ---
for _, row in gdf_espiras.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=f"ID: {row.id_punto_medida} | Ih: {row.vehiculos_por_hora}"
    ).add_to(m)


folium.LayerControl().add_to(m)
m


Cada tramo se vincula con la espira más cercana que se encuentra sobre su trazado, utilizando un algoritmo de correspondencia espacial basado en la geometría y la proximidad.

## Explicación del algoritmo

**Cálculo**

Consideremos el siguiente caso simplificado:



A---C----------B

El objetivo es determinar si el punto **C** (espira) se encuentra sobre el tramo definido por los puntos **A** y **B**.  
Para ello, se verifica la siguiente condición:

$$
dist(A, C) + dist(C, B) \approx dist(A, B)
$$

Si la suma de las distancias desde **A** hasta **C** y desde **C** hasta **B** es igual (dentro de un margen de tolerancia) a la distancia total entre **A** y **B**, concluimos que el punto **C** pertenece al tramo.

---

**Aplicación sobre el conjunto de datos**

El algoritmo recorre cada tramo y evalúa esta condición para todas las espiras.  
Cuando varias espiras cumplen la igualdad (debido al margen de error permitido), se selecciona aquella con la **menor distancia perpendicular al tramo**, es decir, la más cercana al eje de la vía.

El coste computacional del procedimiento es:

$$
O(N \times E)
$$

donde:

- \( N \): número de tramos  
- \( E \): número de espiras


In [ ]:
import numpy as np

def dist_np(lat1, lon1, lat2, lon2):
    """
    Versión vectorizada de Haversine: acepta arrays o escalares.
    Devuelve distancia en metros.
    """
    R = 6371000.0  # radio terrestre en metros
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c


def asignar_espira_a_tramo_np(puntoA, puntoB, espiras, margen_error_metros=1.0):
    lonA, latA = puntoA
    lonB, latB = puntoB

    lat_esp = espiras["latitud"].values
    lon_esp = espiras["longitud"].values
    ids_esp = espiras["identificador"].values

    # distancias vectorizadas
    dist_A_B = dist_np(latA, lonA, latB, lonB)
    dist_A_C = dist_np(latA, lonA, lat_esp, lon_esp)
    dist_C_B = dist_np(lat_esp, lon_esp, latB, lonB)

    errores = np.abs((dist_A_C + dist_C_B) - dist_A_B)

    # aplicar condición de margen
    mask = errores <= margen_error_metros
    if not np.any(mask):
        return None

    idx_best = np.argmin(errores[mask])
    return ids_esp[mask][idx_best]


Aplicamos el algoritmo sobre los diferentes conjuntos de datos

In [ ]:
tqdm.pandas()


inicio = time.time()

print("⏳ Iniciando asignación de espiras a tramos...")

df_osm["espira_id"] = df_osm.apply(
    lambda x: asignar_espira_a_tramo_np(
        (x["lon_A"], x["lat_A"]),
        (x["lon_B"], x["lat_B"]),
        df,
        margen_error_metros=5.0
    ),
    axis=1
)

fin = time.time()
duracion = fin - inicio

# ======================
# 📊 Mostrar resultados
# ======================
print(f"\n✅ Asignación completada en {duracion/60:.2f} minutos "
      f"({duracion:.1f} segundos).")
print(f"Total de tramos procesados: {len(df_osm)}")



In [ ]:
len(df_osm) - df_osm['espira_id'].isna().sum()

In [ ]:
# --- Crear GeoDataFrame de espiras ---
gdf_espiras = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitud, df.latitud),
    crs="EPSG:4326"
)

# --- Crear mapa base ---
m = folium.Map(location=[39.4699, -0.3763], zoom_start=13, tiles="cartodb positron")

# --- Tramos OSM ---
for _, row in df_osm.iterrows():
    if isinstance(row.geometry, LineString):
        # invertir coordenadas lon→lat
        coords = [(lat, lon) for lon, lat in row.geometry.coords]

        # color según si tiene espira asignada
        if pd.notna(row.get("espira_id")) and row["espira_id"] is not None:
            color = "green"   # tiene espira
        else:
            color = "blue"    # no tiene espira

        folium.PolyLine(
            coords,
            color=color,
            weight=2,
            opacity=0.6,
            popup=f"Tramo OSM | Espira: {row.get('espira_id', 'Ninguna')}"
        ).add_to(m)

# --- Espiras ---
for _, row in gdf_espiras.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=f"ID: {row.id_punto_medida} | Ih: {row.vehiculos_por_hora}"
    ).add_to(m)

# --- Control de capas ---
folium.LayerControl().add_to(m)

m


In [ ]:
# Nos quedamos con todos los tramos, aunque no tengan espira
datos_modelo = df_osm.merge(
    df,
    left_on="espira_id",
    right_on="identificador",
    how="left",
    suffixes=("_tramo", "_espira")
)

# --- Guardar resultado ---
output_path = "../data/datos_modelo.csv"
datos_modelo.to_csv(output_path, index=False, encoding="utf-8")

print(f"✅ Merge completado y guardado en: {output_path}")
print(f"Filas totales: {len(datos_modelo)}")
print(f"Columnas totales: {len(datos_modelo.columns)}")
